In [0]:
%sql
use catalog test;
use schema sql;

#### SQL Constraints :
###### 1. NOT NULL --> If any table is created with any column as NOT NULL, then we can't keep that column empty while inserting data.
###### 2. CHECK --> It checks if record presetn in table falls under our mentioned records. If not, it wil show error
###### 3. DEFAULT --> it defaults the value to what we mentioned if its null/not mentioned in table
###### 4. UNIQUE --> It helps to create the table with unique data. Will show error if same data tried to be inserted.

In [0]:
%sql
-- CREATE TABLE IF NOT EXISTS continent (
--     continent STRING NOT NULL,
--     population BIGINT
-- );

-- INSERT INTO continent
-- VALUES
-- (null, 1460481785),
-- ('Asia', 4753082503),
-- ('Americas', 1043901539),
-- ('Europe', 742272147),
-- ('Oceano', 45575773);

In [0]:
%sql
-- CREATE TABLE IF NOT EXISTS continent (
--     continent STRING,
--     population BIGINT
-- );

-- INSERT INTO continent
-- VALUES
-- ('Africa', 1460481785),
-- ('Asia', 4753082503),
-- ('Americas', 1043901539),
-- ('Europe', 742272147),
-- ('Oceano', 45575773);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS continent_incremental (
    continent STRING,
    population BIGINT
);

INSERT INTO continent_incremental
VALUES
('Asia', 5000000000),
('Antarctica', 0);

MERGE INTO continent AS target
USING (
    SELECT DISTINCT continent, population
    FROM continent_incremental
) AS source
ON target.continent = source.continent
WHEN MATCHED THEN
    UPDATE SET
        target.population = source.population
WHEN NOT MATCHED THEN
    INSERT (continent, population)
    VALUES (source.continent, source.population);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [0]:
%sql
select * from continent;

continent,population
Africa,1460481785
Americas,1043901539
Europe,742272147
Oceano,45575773
Asia,5000000000
Antarctica,0


In [0]:
%sql
-- CREATE TABLE IF NOT EXISTS country (
--     country STRING,
--     continent STRING,
--     population BIGINT
-- );

-- INSERT INTO country
-- VALUES
-- ('India', 'Asia', 1428627663),
-- ('China', 'Asia', 1425671352),
-- ('United States', 'Americas', 339996564),
-- ('Indonesia', 'Asia', 277534123),
-- ('Pakistan', 'Asia', 240485658),
-- ('Nigeria', 'Africa', 223804632),
-- ('Brazil', 'Americas', 216422446),
-- ('Bangladesh', 'Asia', 172954319),
-- ('Russia', 'Europe', 144444359),
-- ('Mexico', 'Americas', 128455567);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS country_incremental (
    country STRING PRIMARY KEY,
    continent STRING,
    population BIGINT
);

INSERT INTO country_incremental
VALUES
('India', 'Asia', 1500000000),
('Nepal', 'Asia', 30000000);

MERGE INTO country AS target
USING (
    SELECT DISTINCT
        country,
        continent,
        population
    FROM country_incremental
) AS source
ON target.country = source.country
WHEN MATCHED THEN
    UPDATE SET
        target.continent = source.continent,
        target.population = source.population
WHEN NOT MATCHED THEN
    INSERT (
        country,
        continent,
        population
    )
    VALUES (
        source.country,
        source.continent,
        source.population
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [0]:
%sql
select * from country;

country,continent,population
India,Asia,1500000000
Nepal,Asia,30000000
China,Asia,1425671352
United States,Americas,339996564
Indonesia,Asia,277534123
Pakistan,Asia,240485658
Nigeria,Africa,223804632
Brazil,Americas,216422446
Bangladesh,Asia,172954319
Russia,Europe,144444359


In [0]:
from pyspark.sql.types import *

schema_continent = StructType([
    StructField("continent", StringType(), True),
    StructField("population", LongType(), True)
])

data_continent = [
    ("Africa", 1460481785),
    ("Asia", 4753082503),
    ("Americas", 1043901539),
    ("Europe", 742272147),
    ("Oceano", 45575773)
]

df_continent = spark.createDataFrame(
    data_continent,
    schema_continent
)

df_continent.write \
    .mode("overwrite") \
    .saveAsTable("test.pyspark.continent")

In [0]:
from pyspark.sql.types import *

schema_country = StructType([
    StructField("country", StringType(), True),
    StructField("continent", StringType(), True),
    StructField("population", LongType(), True)
])

data_country = [
    ("India", "Asia", 1428627663),
    ('Nepal', 'Asia', 30000000),
    ("China", "Asia", 1425671352),
    ("United States", "Americas", 339996564),
    ("Indonesia", "Asia", 277534123),
    ("Pakistan", "Asia", 240485658),
    ("Nigeria", "Africa", 223804632),
    ("Brazil", "Americas", 216422446),
    ("Bangladesh", "Asia", 172954319),
    ("Russia", "Europe", 144444359),
    ("Mexico", "Americas", 128455567)
]

df_country = spark.createDataFrame(
    data_country,
    schema_country
)

df_country.write \
    .mode("overwrite") \
    .saveAsTable("test.pyspark.country")

In [0]:
df_continent = spark.table("test.pyspark.continent")
df_country = spark.table("test.pyspark.country")

df_continent.display()
df_country.display()

continent,population
Africa,1460481785
Asia,4753082503
Americas,1043901539
Europe,742272147
Oceano,45575773


country,continent,population
India,Asia,1428627663
Nepal,Asia,30000000
China,Asia,1425671352
United States,Americas,339996564
Indonesia,Asia,277534123
Pakistan,Asia,240485658
Nigeria,Africa,223804632
Brazil,Americas,216422446
Bangladesh,Asia,172954319
Russia,Europe,144444359


#### ALTER TABLE
- RENAME Table
- RENAME Column
- ADD Column
- CHANGE Data Type
- DROP Column

In [0]:
%sql
-- alter table continent rename to test_continent;
-- alter table test_continent rename column population to test_population;
-- alter table test_continent add column testing int;
-- alter table test_continent drop column testing;
-- alter table test_continent alter column test_population type decimal(20,0);

In [0]:
from pyspark.sql.functions import lit, col

a = df_continent.select("*")
a.write.mode("overwrite").saveAsTable("test.pyspark.a")

a.withColumnRenamed("population", "pop").display()
a.withColumn("test", lit(" ")).display()
a.printSchema()
a.dtypes
a.withColumn("population", col("population").cast("decimal(20,0)")).dtypes

continent,pop
Africa,1460481785
Asia,4753082503
Americas,1043901539
Europe,742272147
Oceano,45575773


continent,population,test
Africa,1460481785,
Asia,4753082503,
Americas,1043901539,
Europe,742272147,
Oceano,45575773,


root
 |-- continent: string (nullable = true)
 |-- population: long (nullable = true)



[('continent', 'string'), ('population', 'decimal(20,0)')]